### Detailed code for identified pathways to monitor and measure impact of maternal and child health related interventions in Kakamega, Kenya.

- Complication Pathway
- Prediction Accuracy & Early Intervention Pathway
- Neonatal Health & Birth Outcomes Pathway
- Healthcare Access & Transfer Decision Pathway


In [6]:
# --------------------------------------------------------------
# Setup and Data Loading
# --------------------------------------------------------------

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.colors import qualitative
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df_baseline = pd.read_csv('ind_baseline.csv')
df_scenerio = pd.read_csv('ind_scenerio.csv')

print(f"Baseline data shape: {df_baseline.shape}")
print(f"Scenerio data shape: {df_scenerio.shape}")

Baseline data shape: (406440, 69)
Scenerio data shape: (406440, 69)


In [7]:
# --------------------------------------------------------------
# Utility Functions
# --------------------------------------------------------------
def create_sankey_base(data, stages, stage_labels, title, colors_map=None):
    """
    Base function to create Sankey diagrams
    """
    # Create all unique labels across stages
    all_labels = []
    for stage in stages:
        stage_labels_list = sorted(data[stage].unique().tolist())
        all_labels.extend(stage_labels_list)
    
    # Remove duplicates while preserving order
    unique_labels = []
    seen = set()
    for label in all_labels:
        if label not in seen:
            unique_labels.append(label)
            seen.add(label)
    
    # Create node mapping
    node_dict = {label: i for i, label in enumerate(unique_labels)}
    
    # Create flows between consecutive stages
    source_nodes = []
    target_nodes = []
    values = []
    
    for i in range(len(stages) - 1):
        source_stage = stages[i]
        target_stage = stages[i + 1]
        
        # Calculate flows between stages
        flow_data = data.groupby([source_stage, target_stage]).size().reset_index(name='count')
        
        for _, row in flow_data.iterrows():
            source_label = row[source_stage]
            target_label = row[target_stage]
            count = row['count']
            
            source_nodes.append(node_dict[source_label])
            target_nodes.append(node_dict[target_label])
            values.append(count)
    
    # Calculate node totals
    node_totals = {}
    for stage in stages:
        stage_totals = data[stage].value_counts()
        for label, count in stage_totals.items():
            if label not in node_totals:
                node_totals[label] = count
    
    # Create node labels with totals
    labeled_nodes = []
    for label in unique_labels:
        total = node_totals.get(label, 0)
        labeled_nodes.append(f"{label}\n{total:,}")
    
    # Default colors
    default_colors = [
        'rgba(65, 105, 225, 0.8)',   # Blue
        'rgba(60, 179, 113, 0.8)',   # Green  
        'rgba(220, 20, 60, 0.8)',    # Red
        'rgba(255, 140, 0, 0.8)',    # Orange
        'rgba(138, 43, 226, 0.8)',   # Purple
        'rgba(0, 206, 209, 0.8)',    # Cyan
        'rgba(255, 182, 193, 0.8)',  # Pink
        'rgba(144, 238, 144, 0.8)',  # Light Green
        'rgba(255, 215, 0, 0.8)',    # Gold
        'rgba(128, 128, 128, 0.8)'   # Gray
    ]
    
    # Apply colors
    node_colors = []
    for i, label in enumerate(unique_labels):
        if colors_map and label in colors_map:
            node_colors.append(colors_map[label])
        else:
            node_colors.append(default_colors[i % len(default_colors)])
    
    # Create Sankey diagram
    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=labeled_nodes,
            color=node_colors
        ),
        link=dict(
            source=source_nodes,
            target=target_nodes,
            value=values,
            color='rgba(128, 128, 128, 0.2)',
            # hovertemplate='%{source.label} → %{target.label}<br>Flow: %{value:,}<extra></extra>'
            hovertemplate='%{source.label} → %{target.label}<br>Flow: %{value:,} (%{customdata:.1f}%)<extra></extra>',
            customdata=[round((val / node_totals[unique_labels[src]] * 100), 1) for val, src in zip(values, source_nodes)]
        )
    ))
    
    fig.update_layout(
        title_text=title,
        font_size=10,
        font_color="black",
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=700,
        width=1400
    )
    
    return fig

In [8]:

# --------------------------------------------------------------
# PATHWAY 1: Risk stratification and delivery location pathway
# Mothers > i_risk > i_pocus > i_risk_predicted > i_loc
# --------------------------------------------------------------

def create_risk_delivery_pathway(df, title_suffix=""):
    """
    Create risk stratification and delivery location pathway
    """
    data = df.copy()
    
    # Stage 1: All mothers
    data['Stage1'] = 'Mothers'

    # Stage 2: POCUS Intervention 
    data['Stage2'] = data['i_pocus'].map({0: 'No POCUS', 1: 'POCUS'})
    
    # Stage 3: Actual Risk Stratification
    data['Stage3'] = data['i_risk'].map({0: 'Low Risk', 1: 'High Risk'})

    
    # Stage 4: Predicted Risk Stratification
    data['Stage4'] = data['i_risk_pred'].map({0: 'Low Risk Predicted', 1: 'High Risk Predicted'})

    # Stage 5: Delivery Location
    data['Stage5'] = data['i_loc'].map({0: 'Home', 1: 'L2/L3', 2: 'L4', 3: 'L5'})

    # Define colors for all possible values
    colors_map = {
        'Mothers': 'rgba(65, 105, 225, 0.8)',
        'Low Risk': 'rgba(144, 238, 144, 0.8)',
        'High Risk': 'rgba(255, 140, 0, 0.8)',
        'No POCUS': 'rgba(205, 70, 0, 0.5)',
        'POCUS': 'rgba(255, 70, 0, 0.8)',
        'Low Risk Predicted': 'rgba(60, 179, 113, 0.8)',
        'High Risk Predicted': 'rgba(220, 20, 60, 0.8)',
        'Home': 'rgba(255, 182, 193, 0.8)',
        'L2/L3': 'rgba(0, 206, 209, 0.8)',
        'L4': 'rgba(255, 215, 0, 0.8)',
        'L5': 'rgba(138, 43, 226, 0.4)'
    }
    
    stages = ['Stage1', 'Stage2', 'Stage3', 'Stage4', 'Stage5']
    title = f"Risk Stratification and Delivery Location Pathway{title_suffix}<br>Total Population: {len(data):,}"
    
    return create_sankey_base(data, stages, None, title, colors_map)


In [9]:
# --------------------------------------------------------------
# PATHWAY 2: ANC care, complications, and maternal death pathway
# Mothers > i_ANC > i_anaemia > all actual complications > i_mat_death
# --------------------------------------------------------------

def create_ANC_maternal_death_pathway(df, title_suffix=""):
    """
    Create ANC care, complications, and maternal death pathway
    """
    data = df.copy()
    
    # Stage 1: All mothers
    data['Stage1'] = 'Mothers'
    
    # Stage 2: ANC Status
    data['Stage2'] = data['i_ANC'].map({0: 'No ANC', 1: 'ANC'})
    
    # Stage 3: Anemia Status  
    data['Stage3'] = data['i_anemia'].map({0: 'No Anemia', 1: 'Anemia'})
    
    # Stage 4: Primary Complications (filter to only complications)
    def categorize_complications(row):
        if row['i_pph'] == 1:
            return 'PPH'
        elif row['i_OL'] == 1:
            return 'Obstructed Labor'
        elif row['i_PL'] == 1:
            return 'Prolonged Labor'
        elif row['i_eclampsia'] == 1:
            return 'Eclampsia'
        elif row['i_mat_sepsis'] == 1:
            return 'Sepsis'
        elif row['i_hypoxia'] == 1:
            return 'Hypoxia'
        else:
            return None
    
    data['Stage4'] = data.apply(categorize_complications, axis=1)
    
    # Filter to only cases with complications
    data_complications = data[data['Stage4'].notna()].copy()
    
    # Stage 5: Maternal Death Status
    data_complications['Stage5'] = data_complications['i_mat_death'].map({0: 'Survived', 1: 'Maternal Death'})
    
    # Define colors
    colors_map = {
        'Mothers': 'rgba(65, 105, 225, 0.8)',
        'ANC': 'rgba(60, 179, 113, 0.8)',
        'No ANC': 'rgba(100, 20, 60, 0.8)',
        'Anemia': 'rgba(255, 140, 0, 0.8)',
        'No Anemia': 'rgba(138, 43, 226, 0.8)',
        'PPH': 'rgba(0, 206, 209, 0.8)',
        'Obstructed Labor': 'rgba(255, 182, 193, 0.8)',
        'Prolonged Labor': 'rgba(147, 112, 219, 0.8)',
        'Hypoxia': 'rgba(255, 99, 71, 0.8)',
        'Eclampsia': 'rgba(144, 238, 144, 0.8)',
        'Sepsis': 'rgba(255, 192, 203, 0.8)',
        'Survived': 'rgba(60, 179, 113, 0.8)',
        'Maternal Death': 'rgba(220, 20, 60, 0.8)'
    }
    
    stages = ['Stage1', 'Stage2', 'Stage3', 'Stage4', 'Stage5']
    title = f"ANC Care, Complications, and Maternal Death Pathway{title_suffix}<br>Cases with Complications: {len(data_complications):,}"
    
    return create_sankey_base(data_complications, stages, None, title, colors_map)


In [16]:
# --------------------------------------------------------------
# PATHWAY 3: Intrapartum monitoring and delivery mode pathway
# Mother > i_loc > leading complications > i_sensors > predicted complications > i_transfer_pred > i_mod > i_unnecessary_cs
# --------------------------------------------------------------

def create_intrapartum_pathway(df, title_suffix=""):
    """
    CORRECTED: Intrapartum monitoring and delivery mode pathway
    """
    data = df.copy()
    
    # Stage 1: All mothers
    data['Stage1'] = 'Mothers'

    # Stage 2: Delivery Location
    data['Stage2'] = data['i_loc'].map({0: 'Home', 1: 'L2/L3', 2: 'L4', 3: 'L5'})

    # Stage 3: Sensors Implementation
    data['Stage3'] = data['i_sensors'].map({0: 'No Sensors', 1: 'Sensors'})

    # Stage 4: Leading Complications (actual)
    def get_leading_complication(row):
        if row['i_PL'] == 1:
            return 'Prolonged Labor'
        elif row['i_OL'] == 1:
            return 'Obstructed Labor'
        elif row['i_hypoxia'] == 1:
            return 'Hypoxia'
        else:
            return 'No Leading Complication'
    
    data['Stage4'] = data.apply(get_leading_complication, axis=1)

    

    # Stage 5: Predicted Leading Complications
    def get_predicted_complication(row):
        if row['i_PL_pred'] == 1:
            return 'Prolonged Labor Predicted'
        elif row['i_OL_pred'] == 1:
            return 'Obstructed Labor Predicted'
        elif row['i_hypoxia_pred'] == 1:
            return 'Hypoxia Predicted'
        else:
            return 'No Complication Predicted'
    
    data['Stage5'] = data.apply(get_predicted_complication, axis=1)

    

    # Stage 6: Mode of delivery
    data['Stage6'] = data['i_mod'].map({
        'SVD': 'Spontaneous Vaginal Delivery', 
        'AVD': 'Assisted Vaginal Delivery', 
        'EmCS': 'Emergency Cesarean', 
        'ELCS': 'Elective Cesarean'
    })
    
    # Stage 7: Unnecessary CS
    data['Stage7'] = data['i_unnecessary_cs'].map({0: 'Necessary', 1: 'Unnecessary CS'})

    # Remove any rows with missing values in critical stages
    stages = ['Stage1', 'Stage2', 'Stage3', 'Stage4', 'Stage5', 'Stage6', 'Stage7']
    data = data.dropna(subset=['i_mod'])  # Only drop if mode of delivery is missing
    
    colors_map = {
        'Mothers': 'rgba(65, 105, 225, 0.8)',
        'Home': 'rgba(255, 182, 193, 0.8)',
        'L2/L3': 'rgba(0, 206, 209, 0.8)',
        'L4': 'rgba(255, 215, 0, 0.8)',
        'L5': 'rgba(138, 43, 226, 0.4)',
        'Prolonged Labor': 'rgba(144, 238, 144, 0.8)',
        'Obstructed Labor': 'rgba(255, 140, 0, 0.8)',
        'Hypoxia': 'rgba(255, 99, 71, 0.8)',
        'No Leading Complication': 'rgba(200, 200, 200, 0.8)',
        'No Sensors': 'rgba(180, 180, 180, 0.8)',
        'Sensors': 'rgba(100, 149, 237, 0.8)',
        'Prolonged Labor Predicted': 'rgba(144, 238, 144, 0.6)',
        'Obstructed Labor Predicted': 'rgba(255, 140, 0, 0.6)',
        'Hypoxia Predicted': 'rgba(255, 99, 71, 0.6)',
        'No Complication Predicted': 'rgba(200, 200, 200, 0.6)',
        'No Transfer Predicted': 'rgba(138, 43, 226, 0.8)',
        'Transfer Predicted': 'rgba(0, 206, 209, 0.8)',
        'Necessary': 'rgba(55, 210, 0, 0.8)', 
        'Unnecessary CS': 'rgba(255, 15, 0, 0.8)',
        'Assisted Vaginal Delivery': 'rgba(255, 140, 0, 0.8)',
        'Elective Cesarean': 'rgba(205, 140, 0, 0.8)',
        'Emergency Cesarean': 'rgba(155, 140, 0, 0.8)',
        'Spontaneous Vaginal Delivery': 'rgba(25, 140, 0, 0.8)',
    }
    
    title = f"Intrapartum Monitoring and Delivery Mode Pathway{title_suffix}<br>Total Population: {len(data):,}"
    
    return create_sankey_base(data, stages, None, title, colors_map)



In [11]:
# --------------------------------------------------------------
# PATHWAY 4: NEW SDR delivery location pathway
# Mother > i_ANC > i_loc > i_transfer_pred > i_loc_new_v1 > i_transfer_actual > i_loc_new_v2 > i_comp_death_new > i_mat_death
# --------------------------------------------------------------

def create_SDR_delivery_pathway(df, title_suffix=""):
    """
    NEW: SDR delivery location pathway
    """
    data = df.copy()
    
    # Stage 1: All mothers
    data['Stage1'] = 'Mothers'
    
    # Stage 2: ANC Status
    data['Stage2'] = data['i_ANC'].map({0: 'No ANC', 1: 'ANC'})
    
    # Stage 3: Initial Delivery Location
    data['Stage3'] = data['i_loc'].map({0: 'Home', 1: 'L2/L3', 2: 'L4', 3: 'L5'})
    
    # Stage 4: Transfer Prediction
    data['Stage4'] = data['i_transfer_pred'].map({0: 'No Transfer Predicted', 1: '1st Transfer for Emergency C-Section'})
    
    # Stage 5: New Location after Prediction (if available)
    if 'i_loc_new_v1' in data.columns:
        data['Stage5'] = data['i_loc_new_v1'].map({0: 'Home v1', 1: 'L2/L3 v1', 2: 'L4 v1', 3: 'L5 v1'})
    else:
        # If not available, use the original location
        data['Stage5'] = data['i_loc'].map({0: 'Home v1', 1: 'L2/L3 v1', 2: 'L4 v1', 3: 'L5 v1'})
    
    # Stage 6: Actual Transfer
    data['Stage6'] = data['i_transfer_actual'].map({0: 'No Actual Transfer', 1: '2nd Transfer for Complication Treatment'})
    
    # Stage 7: Final Location (if available)
    if 'i_loc_new_v2' in data.columns:
        data['Stage7'] = data['i_loc_new_v2'].map({0: 'Home Final', 1: 'L2/L3 Final', 2: 'L4 Final', 3: 'L5 Final'})
    else:
        # If not available, use stage 5 location
        data['Stage7'] = data['Stage5'].str.replace(' v1', ' Final')
    
    # Stage 8: Complication Death (if available)
    if 'i_comp_death_new' in data.columns:
        data['Stage8'] = data['i_comp_death_new'].map({0: 'No Complication Death', 1: 'Complication Death'})
    else:
        # Use general complications as proxy
        data['Stage8'] = ((data['i_pph'] == 1) | (data['i_OL'] == 1) | (data['i_eclampsia'] == 1) | 
                         (data['i_mat_sepsis'] == 1)).map({False: 'No Complication Death', True: 'Complication Death'})
    
    # Stage 9: Maternal Death
    data['Stage9'] = data['i_mat_death'].map({0: 'Survived', 1: 'Maternal Death'})
    
    colors_map = {
        'Mothers': 'rgba(65, 105, 225, 0.8)',
        'No ANC': 'rgba(100, 20, 60, 0.8)',
        'ANC': 'rgba(60, 179, 113, 0.8)',
        'Home': 'rgba(255, 182, 193, 0.8)',
        'L2/L3': 'rgba(0, 206, 209, 0.8)',
        'L4': 'rgba(255, 215, 0, 0.8)',
        'L5': 'rgba(138, 43, 226, 0.4)',
        'Home v1': 'rgba(255, 182, 193, 0.7)',
        'L2/L3 v1': 'rgba(0, 206, 209, 0.7)',
        'L4 v1': 'rgba(255, 215, 0, 0.7)',
        'L5 v1': 'rgba(138, 43, 226, 0.3)',
        'Home Final': 'rgba(255, 182, 193, 0.9)',
        'L2/L3 Final': 'rgba(0, 206, 209, 0.9)',
        'L4 Final': 'rgba(255, 215, 0, 0.9)',
        'L5 Final': 'rgba(138, 43, 226, 0.5)',
        'No Transfer Predicted': 'rgba(138, 43, 226, 0.8)',
        '1st Transfer for Emergency C-Section': 'rgba(255, 140, 0, 0.8)',
        'No Actual Transfer': 'rgba(144, 238, 144, 0.8)',
        '2nd Transfer for Complication Treatment': 'rgba(255, 99, 71, 0.8)',
        'No Complication Death': 'rgba(60, 179, 113, 0.8)',
        'Complication Death': 'rgba(255, 140, 0, 0.8)',
        'Survived': 'rgba(60, 179, 113, 0.8)',
        'Maternal Death': 'rgba(220, 20, 60, 0.8)'
    }
    
    stages = ['Stage1', 'Stage2', 'Stage3', 'Stage4', 'Stage5', 'Stage6', 'Stage7', 'Stage8', 'Stage9']
    title = f"SDR Delivery Location Pathway{title_suffix}<br>Total Population: {len(data):,}"
    
    return create_sankey_base(data, stages, None, title, colors_map)



In [17]:
# --------------------------------------------------------------
# RUN ALL PATHWAYS
# --------------------------------------------------------------

print("\n" + "="*80)
print("GENERATING ALL FOUR CORRECTED HEALTH PATHWAYS")
print("="*80)

# PATHWAY 1: Risk stratification and delivery location
print("\n1. Creating Risk Stratification and Delivery Location Pathway...")
fig1_baseline = create_risk_delivery_pathway(df_baseline, " - Baseline")
fig1_scenario = create_risk_delivery_pathway(df_scenerio, " - Scenario")

# PATHWAY 2: ANC care, complications, and maternal death
print("2. Creating ANC Care, Complications, and Maternal Death Pathway...")
fig2_baseline = create_ANC_maternal_death_pathway(df_baseline, " - Baseline")
fig2_scenario = create_ANC_maternal_death_pathway(df_scenerio, " - Scenario")

# PATHWAY 3: Intrapartum monitoring and delivery mode
print("3. Creating CORRECTED Intrapartum Monitoring and Delivery Mode Pathway...")
fig3_baseline = create_intrapartum_pathway(df_baseline, " - Baseline")
fig3_scenario = create_intrapartum_pathway(df_scenerio, " - Scenario")

# PATHWAY 4: NEW SDR delivery location
print("4. Creating NEW SDR Delivery Location Pathway...")
fig4_baseline = create_SDR_delivery_pathway(df_baseline, " - Baseline")
fig4_scenario = create_SDR_delivery_pathway(df_scenerio, " - Scenario")

print("\n" + "="*80)
print("ALL PATHWAYS GENERATED SUCCESSFULLY!")
print("="*80)

# Display all pathways
print("\nDisplaying Pathway 1 - Risk Stratification and Delivery Location:")
fig1_baseline.show()
fig1_scenario.show()

print("\nDisplaying Pathway 2 - ANC Care, Complications, and Maternal Death:")
fig2_baseline.show()
fig2_scenario.show()

print("\nDisplaying Pathway 3 - CORRECTED Intrapartum Monitoring and Delivery Mode:")
fig3_baseline.show()
fig3_scenario.show()

print("\nDisplaying Pathway 4 - NEW SDR Delivery Location:")
fig4_baseline.show()
fig4_scenario.show()


GENERATING ALL FOUR CORRECTED HEALTH PATHWAYS

1. Creating Risk Stratification and Delivery Location Pathway...
2. Creating ANC Care, Complications, and Maternal Death Pathway...
3. Creating CORRECTED Intrapartum Monitoring and Delivery Mode Pathway...
4. Creating NEW SDR Delivery Location Pathway...

ALL PATHWAYS GENERATED SUCCESSFULLY!

Displaying Pathway 1 - Risk Stratification and Delivery Location:



Displaying Pathway 2 - ANC Care, Complications, and Maternal Death:



Displaying Pathway 3 - CORRECTED Intrapartum Monitoring and Delivery Mode:



Displaying Pathway 4 - NEW SDR Delivery Location:
